# Bases Python et pandas pour la qualité des données

**DATE732 · Qualité & intégrité des données · à faire avant le TP (≈ 45 min)**

Ce notebook donne le minimum pour réaliser l'audit NovaCommerce. Chaque partie contient un **exemple** à exécuter,
puis un exercice **À vous**. Toutes les données viennent du dossier `donnees/` du TP.

**Comment l'utiliser**
- Jupyter, VS Code ou [Google Colab](https://colab.research.google.com/) (dans Colab, déposez aussi le dossier `donnees/`).
- `Maj + Entrée` exécute une cellule et passe à la suivante. Exécutez les cellules **dans l'ordre**.
- Une erreur n'est pas grave : lisez la **dernière ligne** du message, elle dit presque toujours ce qui ne va pas.

Pour aller plus loin : [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) ·
[Johns Hopkins, Getting and Cleaning Data](https://www.coursera.org/learn/data-cleaning) ·
[UC Berkeley, Data 100 : Data Cleaning and EDA](https://ds100.org/course-notes-su23/eda/eda.html)

## 1. Python en cinq minutes

Variables, listes, dictionnaires, fonctions : c'est tout ce qu'il faut pour lire le code du TP.

In [ ]:
seuil = 99.0                                   # un nombre
source = 'clients_crm.csv'                     # un texte
statuts = ['ACTIF', 'INACTIF']                 # une liste
regle = {'id': 'R02', 'seuil': 100, 'dimension': 'Unicité'}   # un dictionnaire : clé → valeur

def taux(conformes, controles):
    """Taux de conformité en %, arrondi à 2 décimales."""
    return round(100 * conformes / controles, 2)

print(f"{regle['id']} · {regle['dimension']} : {taux(15292, 15412)} % (seuil {regle['seuil']} %)")
'INACTIF' in statuts

**À vous** · écrivez une fonction `conforme(taux, seuil)` qui renvoie le texte `'conforme'` ou `'non conforme'`, puis testez-la avec 99.22 et 100.

In [ ]:
# Votre code ici
# Indice : if ... else ... sur une seule ligne, ou un bloc if / else

## 2. Lire un fichier sans l'abîmer

Par défaut, pandas **devine** le type de chaque colonne. Sur des données de qualité douteuse, c'est dangereux :
il supprime les zéros initiaux, transforme des codes en nombres, des vides en `NaN`.
Pour auditer, on lit tout **en texte** et on convertit ensuite, colonne par colonne, en comptant ce qui échoue.

In [ ]:
import pandas as pd
from pathlib import Path
D = Path('donnees')

devine = pd.read_csv(D / 'clients_crm.csv')                                  # pandas devine les types
texte = pd.read_csv(D / 'clients_crm.csv', dtype=str, keep_default_na=False)  # tout en texte, vides gardés

print('type deviné pour code_postal :', devine['code_postal'].dtype)
print('lu en texte                  :', texte['code_postal'].dtype)
devine['code_postal'].head(3).tolist(), texte['code_postal'].head(3).tolist()

Le code postal de Nice (06000) est devenu `6000` dans la première lecture : l'outil a créé une anomalie. **Dans tout le TP, lisez en texte.**

## 3. Regarder avant de mesurer

`shape`, `head`, `columns`, `value_counts`, `describe` : le profiling commence toujours ici.

In [ ]:
produits = pd.read_csv(D / 'produits.csv', dtype=str, keep_default_na=False)
print(produits.shape)            # (lignes, colonnes)
print(list(produits.columns))
produits.head()

In [ ]:
# Les valeurs d'une colonne et leur fréquence : le meilleur détecteur d'anomalies de format
produits['chaine_du_froid'].value_counts(dropna=False)

**À vous** · affichez les valeurs de la colonne `categorie` de `produits` et leur fréquence. Comparez avec `referentiels.csv` : que remarquez-vous ?

In [ ]:
# Votre code ici
# Indice : value_counts(), puis lire referentiels.csv et filtrer type_referentiel == 'CATEGORIE_PRODUIT'

## 4. Sélectionner et filtrer

- `df['col']` : une colonne ; `df[['a', 'b']]` : plusieurs colonnes.
- `df[condition]` : les lignes où la condition est vraie. Combiner : `&` (et), `|` (ou), `~` (non), **avec des parenthèses**.
- Sur du texte : `.str.strip()`, `.str.upper()`, `.str.contains(motif)`, `.str.match(motif)`, `.isin(liste)`.

In [ ]:
froid = produits[produits['chaine_du_froid'] == 'oui']
print(len(froid), 'produits sous chaîne du froid')

autorisees = ['EPICERIE', 'FRAIS', 'BOISSONS', 'MAISON']
hors_liste = produits[~produits['categorie'].isin(autorisees)]
print(len(hors_liste), 'produits hors référentiel avant nettoyage')
print(len(produits[~produits['categorie'].str.strip().str.upper().isin(autorisees)]), 'après normalisation (strip + upper)')

**À vous** · dans `clients_crm.csv` (variable `texte`), comptez les e-mails vides, puis les e-mails non vides qui ne respectent pas le motif `^[^@\s]+@[^@\s]+\.[a-z]{2,}$`.

In [ ]:
# Votre code ici
# Indice : comparez à '' pour les vides ; .str.match(motif) renvoie True / False ; ~ inverse

## 5. Convertir en comptant les échecs

`pd.to_numeric(..., errors='coerce')` et `pd.to_datetime(..., errors='coerce')` mettent `NaN` / `NaT`
quand la conversion échoue. **Comptez ces échecs** : ce sont des anomalies de format.

In [ ]:
prix = pd.to_numeric(produits['prix_ht_eur'], errors='coerce')
print('prix non convertibles :', prix.isna().sum())
prix.describe()

In [ ]:
creation = pd.to_datetime(texte['date_creation'], format='%Y-%m-%d', errors='coerce')
echecs = creation.isna() & (texte['date_creation'] != '')
print('dates de création non conformes au format AAAA-MM-JJ :', echecs.sum())
texte.loc[echecs, 'date_creation'].head(8).tolist()

**À vous** · convertissez `stock` de `produits` en nombre. Combien de valeurs sont négatives ? Lesquelles sont nulles ? Une valeur nulle est-elle une anomalie ?

In [ ]:
# Votre code ici
# Indice : pd.to_numeric(..., errors='coerce'), puis (stock < 0).sum()

## 6. Doublons et regroupements

- `df.duplicated(subset=[...], keep=False)` marque **toutes** les lignes d'une clé dupliquée.
- `df[col].nunique()` compte les valeurs distinctes.
- `df.groupby(col).size()` ou `.agg(...)` : compter ou résumer par groupe.

Un doublon se décrit avec **deux nombres** : les lignes concernées et les clés distinctes concernées.

In [ ]:
dup = produits.duplicated(subset=['produit_id'], keep=False)
print('lignes concernées :', dup.sum(), '· identifiants distincts concernés :', produits.loc[dup, 'produit_id'].nunique())
produits[dup].sort_values('produit_id')[['produit_id', 'libelle', 'prix_ht_eur', 'taux_tva']]

In [ ]:
produits.assign(prix=prix).groupby('chaine_du_froid')['prix'].agg(['count', 'mean', 'max']).round(2)

**À vous** · dans `clients_crm.csv`, combien d'identifiants `client_id` sont dupliqués (lignes et identifiants distincts) ? Puis comptez les fiches par `canal_creation`.

In [ ]:
# Votre code ici
# Indice : duplicated(subset=['client_id'], keep=False) ; groupby('canal_creation').size()

## 7. Joindre sans multiplier

`gauche.merge(droite, on='cle', how='left', indicator=True, validate='many_to_one')` :
- `indicator=True` ajoute la colonne `_merge` (`both`, `left_only`) : pratique pour trouver les **orphelins** ;
- `validate=` **refuse** la jointure si la cardinalité n'est pas celle attendue : c'est un contrôle qualité en une ligne.

L'exemple ci-dessous est volontairement minuscule : une commande payée en deux fois.

In [ ]:
commandes_jouet = pd.DataFrame({'commande_id': ['CMD1', 'CMD2'], 'montant': [100.0, 50.0]})
paiements_jouet = pd.DataFrame({'commande_id': ['CMD1', 'CMD1', 'CMD2'], 'paye': [60.0, 40.0, 50.0]})

j = commandes_jouet.merge(paiements_jouet, on='commande_id', how='left')
print('somme des commandes :', commandes_jouet['montant'].sum(), '· après jointure :', j['montant'].sum())
j

In [ ]:
try:
    commandes_jouet.merge(paiements_jouet, on='commande_id', validate='one_to_one')
except Exception as e:
    print('Jointure refusée :', e)

# La bonne méthode : agréger d'abord le côté « plusieurs », puis joindre en 1 pour 1
paye = paiements_jouet.groupby('commande_id', as_index=False)['paye'].sum()
commandes_jouet.merge(paye, on='commande_id', validate='one_to_one')

**À vous** · joignez `avis_clients.csv` à `produits` sur `produit_id` avec `indicator=True`. Combien d'avis portent sur un produit inconnu ? Attention : `produits` a-t-il une clé unique ?

In [ ]:
# Votre code ici
# Indice : how='left', indicator=True ; si validate='many_to_one' échoue, regardez la section 6

## 8. Dates et fuseaux horaires

Une date sans fuseau est **ambiguë**. Dans le TP, certaines sources sont en UTC (`Z`), d'autres en heure de Paris.
`pd.to_datetime(..., utc=True)` ramène tout en UTC ; `tz_convert('Europe/Paris')` repasse en heure locale.

In [ ]:
s = pd.Series(['2026-03-31T21:59:30Z', '2026-03-31T23:59:30+02:00', '2026-03-31 23:59:30'])
utc = pd.to_datetime(s, utc=True, format='mixed', errors='coerce')
pd.DataFrame({'texte': s, 'UTC': utc, 'Paris': utc.dt.tz_convert('Europe/Paris')})
# Les deux premières sont le même instant. La troisième n'a pas de fuseau : pandas la suppose en UTC… à tort ?

In [ ]:
# Un même instant peut changer de mois selon le fuseau : à quel mois rattacher ce paiement ?
utc.dt.tz_convert('Europe/Paris').dt.month.tolist(), utc.dt.month.tolist()

**À vous** · le 25 octobre 2026, la France repasse à l'heure d'hiver. Créez les horodatages locaux toutes les 30 min de 01:00 à 04:00 ce jour-là avec `pd.date_range(..., tz='Europe/Paris')`. Que remarquez-vous à 02:00–03:00 ?

In [ ]:
# Votre code ici
# Indice : partez d'un range en UTC puis tz_convert('Europe/Paris'), et affichez .strftime('%H:%M')

## 9. Expressions régulières : découper une ligne de log

Un log est du texte. Une expression régulière avec des **groupes nommés** `(?P<nom>...)` le transforme en colonnes.
Documentation : [module re](https://docs.python.org/fr/3/library/re.html) · testeur en ligne : [regex101](https://regex101.com/).

In [ ]:
import re
ligne = ('2026-10-05T08:12:03Z INFO psp-gateway event_id=EV0001234 event_time=2026-10-05T08:12:01Z '
         'type=PAYMENT_ACCEPTED commande_id=CMD123 montant=42.50 devise=EUR moyen=CB')
motif = re.compile(r'event_id=(?P<event_id>\S+) event_time=(?P<event_time>.+?) type=(?P<type>\S+) '
                   r'commande_id=(?P<commande_id>\S+) montant=(?P<montant>\S*)')
m = motif.search(ligne)
m.groupdict()

**À vous** · lisez les 2 000 premières lignes de `paiements.log`. Combien correspondent au motif ? Affichez trois lignes qui n'y correspondent pas : sont-elles des paiements ?

In [ ]:
# Votre code ici
# Indice : open(...) puis une boucle for ; motif.search(l) renvoie None si la ligne ne correspond pas

## 10. JSON et XML : aplatir des données imbriquées

- JSON : `pd.json_normalize(données)` met une commande par ligne ; `record_path='lignes'` met **un article** par ligne.
- XML : `xml.etree.ElementTree` parcourt les balises ; `.get('attribut')` et `.findtext('balise')` lisent les valeurs.

Documentation : [pandas.json_normalize](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html) ·
[xml.etree.ElementTree](https://docs.python.org/fr/3/library/xml.etree.elementtree.html)

In [ ]:
import json
exemple = [{'commande_id': 'CMD1', 'client_id': 'C1', 'lignes': [{'produit_id': 'PR1', 'quantite': 2}, {'produit_id': 'PR2', 'quantite': 1}]},
           {'commande_id': 'CMD2', 'customerId': 'C2', 'lignes': [{'produit_id': 'PR1', 'quantite': 5}]}]
display(pd.json_normalize(exemple))
pd.json_normalize(exemple, record_path='lignes', meta=['commande_id'])
# Remarquez : la 2e commande n'a pas de client_id mais un customerId. Que donnera un contrôle « client_id renseigné » ?

In [ ]:
import xml.etree.ElementTree as ET
xml = '<factures><facture numero="F1"><total_ttc>12.00</total_ttc></facture><facture><total_ttc>8.40</total_ttc></facture></factures>'
pd.DataFrame([{'numero': f.get('numero'), 'total_ttc': f.findtext('total_ttc')} for f in ET.fromstring(xml)])

**À vous** · chargez `commandes_api.json`. Combien de commandes par `schema_version` ? Pour chaque version, quelles colonnes sont vides ?

In [ ]:
# Votre code ici
# Indice : json.load, pd.json_normalize, value_counts, puis groupby('schema_version') et isna().sum()

## Récapitulatif

| Besoin | Outil |
|---|---|
| Lire sans abîmer | `pd.read_csv(..., dtype=str, keep_default_na=False)` |
| Profiler | `shape`, `value_counts()`, `describe()`, `nunique()` |
| Filtrer | `df[condition]`, `&`, `|`, `~`, `isin`, `.str.match` |
| Convertir | `pd.to_numeric` / `pd.to_datetime` avec `errors='coerce'` et **compter les échecs** |
| Doublons | `duplicated(subset=..., keep=False)` : lignes **et** clés distinctes |
| Joindre | `merge(..., indicator=True, validate=...)`, agréger avant de joindre |
| Dates | `utc=True`, `tz_convert('Europe/Paris')` |
| Logs | `re.compile` avec groupes nommés, compter les lignes rejetées |
| JSON / XML | `pd.json_normalize`, `ElementTree` |

Vous êtes prêts pour `audit_novacommerce.ipynb`.